# Análisis Exploratorio: US GDP vs Public Debt (1947–2020)

**Autor:** [Tu nombre]  
**Dataset:** GDP y Deuda Pública de EE.UU. — series trimestrales (FRED / Depto. del Tesoro)  
**Prerequisito:** ejecutar `01_limpieza.ipynb` primero para generar los CSVs limpios.

---

## Preguntas que responde este análisis

1. ¿Cómo ha crecido el GDP de EE.UU. a lo largo de las décadas?
2. ¿En qué momentos históricos el GDP se contrajo (recesiones)?
3. ¿Cómo evolucionó el ratio Deuda/GDP desde 1966?
4. ¿Qué décadas tuvieron el mayor crecimiento económico promedio?
5. ¿Cuál fue el impacto de la crisis del 2008 y el COVID-19 comparado con recesiones anteriores?


---
## 1. Importar librerías y cargar datos limpios

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Estilo de gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.titlesize": 14,
})

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Cargar datasets limpios generados en 01_limpieza.ipynb
df          = pd.read_csv("data/gdp_debt_clean.csv",    parse_dates=["Quarter"])
df_complete = pd.read_csv("data/gdp_debt_complete.csv", parse_dates=["Quarter"])

print(f"df (completo):   {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"df_complete:     {df_complete.shape[0]} filas × {df_complete.shape[1]} columnas")
df.head()


---
## 2. Visión general del dataset limpio


In [ ]:
# Resumen estadístico de las variables principales
resumen = df_complete[["GDP ($mil)", "Debt ($mil)", "GDP_growth_pct", "Debt_to_GDP_pct"]].describe()
resumen


In [ ]:
# Recesiones identificadas en la serie completa
recesiones = df[df["Recession"] == True][["Quarter", "Year", "Q", "GDP_growth_pct"]]
print(f"Quarters de recesión detectados: {len(recesiones)}")
print()
print(recesiones.to_string(index=False))


---
## 3. ¿Cómo ha crecido el GDP a lo largo del tiempo?

El PIB nominal de EE.UU. creció de ~$243 mil millones en 1947 a ~$21.7 billones en 2020, un aumento de casi 90x. Sin embargo, parte de ese crecimiento refleja inflación. En este análisis trabajamos con cifras nominales tal como vienen del dataset.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(df["Quarter"], df["GDP ($mil)"] / 1e6,
                alpha=0.15, color="#378ADD")
ax.plot(df["Quarter"], df["GDP ($mil)"] / 1e6,
        color="#185FA5", linewidth=1.5, label="GDP nominal")

# Marcar recesiones con franjas verticales
recesion_quarters = df[df["Recession"] == True]["Quarter"]
for q in recesion_quarters:
    ax.axvspan(q - pd.DateOffset(months=3), q, alpha=0.12, color="#E24B4A", linewidth=0)

# Anotaciones de eventos clave
eventos = {
    "1973-10-01": ("Crisis
petróleo",  0.4),
    "2008-10-01": ("Crisis
2008",      9.5),
    "2020-04-01": ("COVID-19",         18.5),
}
for fecha, (label, y_pos) in eventos.items():
    ax.annotate(label, xy=(pd.Timestamp(fecha), y_pos),
                fontsize=8, color="#A32D2D", ha="center",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#F09595", lw=0.5))

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}T"))
ax.set_title("GDP nominal de EE.UU. (1947–2020)")
ax.set_xlabel("")
ax.set_ylabel("Billones (trillions) de USD")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("data/gdp_historico.png", bbox_inches="tight")
plt.show()
print("Las franjas rojas indican quarters de recesión.")


---
## 4. ¿Qué tan severas fueron las recesiones?

Comparamos la caída del GDP en cada recesión moderna usando el crecimiento trimestral.


In [ ]:
# Mínimo crecimiento trimestral por período de recesión agrupado
crisis = {
    "Recesión 1974–75":  ("1973-10-01", "1975-04-01"),
    "Recesión 1980–82":  ("1980-01-01", "1982-10-01"),
    "Recesión 1990–91":  ("1990-07-01", "1991-04-01"),
    "Dot-com 2001":      ("2001-01-01", "2001-10-01"),
    "Crisis 2008–09":    ("2008-07-01", "2009-07-01"),
    "COVID-19 2020":     ("2020-01-01", "2020-07-01"),
}

rows = []
for nombre, (inicio, fin) in crisis.items():
    mask = (df["Quarter"] >= inicio) & (df["Quarter"] <= fin)
    peor_quarter = df[mask]["GDP_growth_pct"].min()
    rows.append({"Período": nombre, "Peor quarter (% cambio GDP)": round(peor_quarter, 2)})

comparativa = pd.DataFrame(rows).sort_values("Peor quarter (% cambio GDP)")

fig, ax = plt.subplots(figsize=(9, 4))
colores = ["#E24B4A" if v < -5 else "#EF9F27" if v < -1 else "#888780"
           for v in comparativa["Peor quarter (% cambio GDP)"]]
bars = ax.barh(comparativa["Período"],
               comparativa["Peor quarter (% cambio GDP)"],
               color=colores, height=0.55)

for bar, val in zip(bars, comparativa["Peor quarter (% cambio GDP)"]):
    ax.text(val - 0.1, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", ha="right", fontsize=9, color="white", fontweight="bold")

ax.axvline(0, color="#444", linewidth=0.8)
ax.set_title("Peor quarter de cada recesión (crecimiento del GDP)")
ax.set_xlabel("Variación trimestral del GDP (%)")
plt.tight_layout()
plt.savefig("data/recesiones_comparativa.png", bbox_inches="tight")
plt.show()
print(comparativa.to_string(index=False))


---
## 5. ¿Cómo evolucionó el ratio Deuda/GDP desde 1966?

El ratio Deuda/GDP es el indicador macroeconómico más usado para evaluar la sostenibilidad fiscal de un país. Un ratio creciente indica que la deuda crece más rápido que la economía.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(df_complete["Quarter"], df_complete["Debt_to_GDP_pct"],
                alpha=0.12, color="#D85A30")
ax.plot(df_complete["Quarter"], df_complete["Debt_to_GDP_pct"],
        color="#993C1D", linewidth=1.8)

# Línea de referencia al 100%
ax.axhline(100, color="#E24B4A", linewidth=0.8, linestyle="--", alpha=0.6)
ax.text(df_complete["Quarter"].iloc[-1], 101, "100%", fontsize=8,
        color="#E24B4A", va="bottom")

# Puntos de inflexión clave
puntos = {
    "1981": ("1981-01-01", "Reaganomics
(recortes impuesto)"),
    "2009": ("2009-01-01", "Crisis 2008
(estímulo fiscal)"),
    "2020": ("2020-04-01", "COVID-19
(135%)"),
}
for key, (fecha, label) in puntos.items():
    row = df_complete[df_complete["Quarter"] == fecha]
    if not row.empty:
        y = row["Debt_to_GDP_pct"].values[0]
        ax.annotate(label, xy=(pd.Timestamp(fecha), y),
                    xytext=(pd.Timestamp(fecha) - pd.DateOffset(years=3), y + 12),
                    fontsize=8, color="#712B13", ha="center",
                    arrowprops=dict(arrowstyle="-", color="#D85A30", lw=0.8))

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.set_title("Ratio Deuda/GDP de EE.UU. (1966–2020)")
ax.set_xlabel("")
ax.set_ylabel("Deuda como % del GDP")
plt.tight_layout()
plt.savefig("data/deuda_gdp_ratio.png", bbox_inches="tight")
plt.show()


---
## 6. ¿Qué décadas tuvieron el mayor crecimiento económico?


In [ ]:
# Crecimiento promedio trimestral del GDP por década
por_decada = (df.groupby("Decade")["GDP_growth_pct"]
                .agg(["mean", "std", "count"])
                .rename(columns={"mean": "Crecimiento promedio (%)",
                                 "std":  "Desviación estándar",
                                 "count": "Quarters"})
                .round(2))

print(por_decada.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

decadas = por_decada.index.tolist()
medias  = por_decada["Crecimiento promedio (%)"].values
stds    = por_decada["Desviación estándar"].values

colores = ["#185FA5" if m >= 2 else "#EF9F27" if m >= 1 else "#E24B4A" for m in medias]
bars = ax.bar(decadas, medias, color=colores, width=0.55,
              yerr=stds, capsize=4, error_kw={"linewidth": 0.8, "color": "#888"})

for bar, val in zip(bars, medias):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f"{val:.2f}%", ha="center", va="bottom", fontsize=9)

ax.axhline(0, color="#444", linewidth=0.8)
ax.set_title("Crecimiento promedio trimestral del GDP por década")
ax.set_ylabel("Variación promedio (%)")
ax.set_xlabel("Década")
plt.tight_layout()
plt.savefig("data/crecimiento_por_decada.png", bbox_inches="tight")
plt.show()
print("Las barras de error representan la desviación estándar (volatilidad del crecimiento).")


---
## 7. Crisis 2008 vs COVID-19: comparativa trimestre a trimestre

Ambas crisis fueron las más severas desde la Gran Depresión, pero con características muy distintas: la de 2008 fue gradual y prolongada; la de 2020 fue instantánea y abrupta.


In [ ]:
# Extraer 8 quarters antes y después del pico de cada crisis
def get_crisis_window(df, peak_date, n_quarters=8):
    idx = df[df["Quarter"] == peak_date].index[0]
    window = df.iloc[max(0, idx-n_quarters): idx+n_quarters+1].copy()
    window["t"] = range(-n_quarters, len(window) - n_quarters)
    return window

crisis_2008 = get_crisis_window(df, "2008-10-01", n_quarters=6)
crisis_2020 = get_crisis_window(df, "2020-04-01", n_quarters=6)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, crisis, titulo, color in zip(
    axes,
    [crisis_2008, crisis_2020],
    ["Crisis financiera 2008", "COVID-19 2020"],
    ["#185FA5", "#E24B4A"]
):
    colores_bar = [color if v < 0 else "#B4B2A9" for v in crisis["GDP_growth_pct"]]
    ax.bar(crisis["t"], crisis["GDP_growth_pct"], color=colores_bar, width=0.7)
    ax.axhline(0, color="#444", linewidth=0.8)
    ax.axvline(0, color=color, linewidth=1, linestyle="--", alpha=0.5)
    ax.set_title(titulo)
    ax.set_xlabel("Quarters desde el peor trimestre")
    ax.set_ylabel("Variación GDP (%)" if ax == axes[0] else "")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))

plt.suptitle("Comparativa de caída del GDP: 2008 vs COVID-19", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("data/crisis_comparativa.png", bbox_inches="tight")
plt.show()


---
## 8. Conclusiones

### Hallazgos principales

**Crecimiento histórico:**
El GDP nominal de EE.UU. creció ~90x entre 1947 y 2020. Las décadas de los 50s y 60s registraron el mayor crecimiento trimestral promedio (>2%), impulsado por la reconstrucción de posguerra y el boom industrial.

**Recesiones:**
El COVID-19 generó la caída trimestral más severa de toda la serie histórica (-9.0% en 2020 Q2), casi el triple que el peor quarter de la crisis de 2008 (-2.8%). Sin embargo, la recuperación post-COVID fue también la más rápida: el GDP rebotó con fuerza en el trimestre siguiente.

**Ratio Deuda/GDP:**
El ratio se mantuvo por debajo del 50% hasta principios de los 1980s. A partir de la política fiscal expansiva de Reagan escaló sostenidamente. Superó el 100% por primera vez en 2013 y llegó a 135% en 2020 Q2 — el nivel más alto de la serie histórica.

**Patrón clave:**
Cada crisis desde 1980 dejó el ratio Deuda/GDP en un piso más alto que el anterior, sin que nunca haya retornado a niveles pre-crisis. Esto sugiere una tendencia estructural de acumulación de deuda en períodos de estrés económico.

### Próximos pasos
- `03_visualizacion.ipynb` — Dashboard interactivo con las métricas clave del proyecto.
